# Chapter 11 -- 第四个分类器：双向 LSTM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaosfrey-arch/news-classification-tutorial/blob/main/chapter11_bilstm.ipynb)

**本章目标：** 理解 RNN 和 LSTM 的核心思想，搭建双向 LSTM 分类器，与 CNN 对比。

**预计时间：** 60 分钟

> 强烈推荐：[Colah's Blog -- Understanding LSTMs](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)（最清晰的 LSTM 图解）

## 11.1 RNN：处理序列数据的网络

CNN 有个局限：它用固定大小的窗口（3-5 个词）看文本，**无法捕获长距离的依赖**。

例如：
> "The player who joined the team last year **scored** a crucial goal."

"player" 和 "scored" 相距很远，CNN 的小窗口可能捕获不到这种关系。

**RNN（循环神经网络）** 的设计理念：按序列顺序处理，每一步都「记住」前面的信息。

```
词1 → [RNN 单元] → 隐藏状态 h1 ─┐
词2 → [RNN 单元] ← h1           │  每个时间步都把上一步的
词3 → [RNN 单元] ← h2           │  隐藏状态传递下去
...                              │
词n → [RNN 单元] → 最终输出     ─┘
```

最终的隐藏状态理论上包含了整个序列的信息。

## 11.2 梯度消失问题与 LSTM

**朴素 RNN 的问题：梯度消失（Vanishing Gradient）**

反向传播时，梯度需要从最后一个时间步传回第一个。经过多次乘法，梯度越来越小，最终接近 0——早期时间步的词对训练几乎没有影响。

**LSTM（Long Short-Term Memory，1997）** 解决了这个问题，引入三个「门控」机制：

| 门 | 作用 |
|----|------|
| 遗忘门（Forget Gate）| 决定丢弃哪些过去信息 |
| 输入门（Input Gate） | 决定记住哪些新信息 |
| 输出门（Output Gate）| 决定输出哪些信息 |

类比：你在读一篇文章，读到新段落时，你会忘记不相关的细节（遗忘门），记住新的关键信息（输入门），并决定现在要用哪些信息来理解当前内容（输出门）。

LSTM 额外维护了一个**细胞状态（Cell State）**，像一条「高速公路」传递长距离信息，梯度可以直接通过它流动，不会消失。

## 11.3 双向 LSTM（BiLSTM）

单向 LSTM 只从左到右读文章，但文字的含义有时需要看**后面的词**：

> "The **bank** can guarantee deposits will eventually cover future tuition costs."
> "She sat on the **bank** of the river."

同一个词 "bank"，看后面的词才知道含义（金融 vs. 河岸）。

**双向 LSTM** 同时从两个方向处理序列：
```
→ 正向 LSTM：词1 → 词2 → ... → 词n  （捕获左到右的上下文）
← 反向 LSTM：词n → 词n-1 → ... → 词1（捕获右到左的上下文）

两个方向的输出拼接 → 每个位置有完整的双向上下文
```

In [ ]:
import pandas as pd, numpy as np, re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'http\S+|www\S+|https\S+', '', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df = pd.read_csv('dataset.csv').dropna(subset=['Class']).copy()
df['Description'] = df['Description'].fillna('')
df['text'] = (df['Title'] + ' ' + df['Description']).apply(clean_text)

le = LabelEncoder()
y = le.fit_transform(df['Class'])
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text'].values, y, test_size=0.3, random_state=42, stratify=y)

VOCAB_SIZE = 20000
MAX_LEN    = 100
EMBED_DIM  = 128

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train_text),
                             maxlen=MAX_LEN, padding='pre', truncating='pre')
X_test_pad  = pad_sequences(tokenizer.texts_to_sequences(X_test_text),
                             maxlen=MAX_LEN, padding='pre', truncating='pre')

y_train_cat = to_categorical(y_train, 4)
y_test_cat  = to_categorical(y_test,  4)
print('数据准备完毕！')

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

LSTM_UNITS = 128

bilstm_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN, name='embedding'),
    Bidirectional(LSTM(LSTM_UNITS,
                       return_sequences=False,   # 只返回最后一步的输出
                       dropout=0.2,              # 输入 Dropout
                       recurrent_dropout=0.2),   # 循环连接 Dropout
                  name='bilstm'),
    Dense(64, activation='relu', name='fc'),
    Dropout(0.5, name='dropout'),
    Dense(4, activation='softmax', name='output')
], name='BiLSTM')

bilstm_model.compile(optimizer='adam',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])
bilstm_model.summary()
print(f'\nBiLSTM 总参数量：{bilstm_model.count_params():,}')

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = bilstm_model.fit(
    X_train_pad, y_train_cat,
    epochs=15,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)
print(f'实际训练了 {len(history.history["loss"])} 轮')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns

# 训练曲线
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, (metric, title) in enumerate([('accuracy', 'Accuracy'), ('loss', 'Loss')]):
    axes[i].plot(history.history[metric], label='Train')
    axes[i].plot(history.history[f'val_{metric}'], label='Validation')
    axes[i].set_title(f'BiLSTM -- {title}', fontsize=13)
    axes[i].set_xlabel('Epoch'); axes[i].set_ylabel(title)
    axes[i].legend(); axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 评估
y_pred_prob = bilstm_model.predict(X_test_pad, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average='macro')
print(f'BiLSTM  Accuracy: {acc:.4f}  |  Macro F1: {f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('BiLSTM -- Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# 保存模型
import joblib
bilstm_model.save('bilstm_model.keras')
joblib.dump(tokenizer, 'tokenizer.pkl')  # 同一个 tokenizer
print('已保存：bilstm_model.keras, tokenizer.pkl')

## CNN vs BiLSTM 对比

| | TextCNN | BiLSTM |
|---|---|---|
| **核心机制** | 局部特征（n-gram 短语）| 全局序列上下文 |
| **速度** | 快（卷积可并行）| 较慢（LSTM 顺序计算）|
| **参数量** | 较少 | 较多 |
| **优势** | 捕获关键词短语 | 理解长距离依赖 |
| **适合场景** | 主题分类 | 情感分析、阅读理解 |

对于新闻分类这种任务，**CNN 和 BiLSTM 效果相近**（都在 91-93% 左右），因为新闻类别主要由关键词决定，长距离依赖不是瓶颈。

## 总结

| 概念 | 含义 |
|------|------|
| RNN | 按序列处理，传递隐藏状态 |
| 梯度消失 | 反向传播时梯度随时间步衰减到 0 |
| LSTM | 用门控机制解决梯度消失，支持长距离记忆 |
| 双向 LSTM | 同时从左到右和从右到左处理序列 |
| return_sequences | False = 只取最后一步输出，True = 输出每一步 |

**下一章 →** Chapter 12：汇总、保存与部署